In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


**CWRU dataset:** Smith, W. A., & Randall, R. B. (2015). Rolling element bearing diagnostics using the Case Western Reserve University data: A benchmark study. *Mechanical Systems and Signal Processing*, 64–65, 100–131.


# 특징 추출 실습: CWRU 베어링 내륜 결함 진단

FFT, 필터링, 시간-주파수 분석, 포락선 분석의 도구를 **하나의 실제 데이터**(CWRU 베어링 가속도 신호)에 차례로 적용하여 파이프라인으로 연결한다.

**목표**
- 정상 신호(`v_n`)와 내륜 결함(inner race, IR) 신호(`v_f`)를 비교한다.
- 시간 영역, 주파수 영역, 공진 대역의 분리, 포락선 복조(demodulation)의 순서로 단계를 따라간다.
- 각 단계에서 정상 신호와 결함 신호의 차이가 어떻게 뚜렷해지는지 관찰하고, 마지막에 그 차이를 **특징 벡터**(feature vector)로 정량화한다.

**데이터 출처**: Case Western Reserve University Bearing Data Center 의 Drive-End 가속도 신호(내륜 결함 1772 rpm, 정상 약 1796 rpm, 샘플링 주파수 12 kHz).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import utils


## 실습 1. 베어링 특성 주파수 계산

베어링에 결함이 발생하면 **축이 1회전하는 동안 볼이 결함부를 지나가는 횟수**가 이론적으로 정해진다. CWRU Drive-End 베어링(SKF 6205)의 기하 상수는 다음과 같다.

| 결함 위치 | 특성 주파수(축 회전 주파수의 배수) |
| --- | --- |
| 내륜 (Inner Race, BPFI) | 5.4152 |
| 외륜 (Outer Race, BPFO) | 3.5848 |

이 배수에 축 회전 주파수 $f_{shaft} = \text{rpm}/60$ 을 곱하면 Hz 단위의 특성 주파수를 얻는다. 이 값이 **포락선 스펙트럼에서 성분이 나타날 것으로 기대되는 주파수**이다.

In [ ]:
rpm = 1772
F_shaft = rpm / 60
print('Shaft frequency:', F_shaft, 'Hz')

# Drive-End 베어링 특성 주파수
DE_BPFI = 5.4152 * F_shaft
DE_BPFO = 3.5848 * F_shaft
print('DE Inner Race Fault Frequency (BPFI):', DE_BPFI, 'Hz')
print('DE Outer Race Fault Frequency (BPFO):', DE_BPFO, 'Hz')

# Fan-End 베어링 (참고)
FE_BPFI = 4.9469 * F_shaft
FE_BPFO = 3.0530 * F_shaft
print('FE Inner Race Fault Frequency:', FE_BPFI, 'Hz')
print('FE Outer Race Fault Frequency:', FE_BPFO, 'Hz')


**관찰**
- 축 회전 주파수는 약 **29.5 Hz** 이고, DE 베어링의 내륜 결함 주파수는 **BPFI ≈ 160 Hz** 이다.
- 이 노트북은 내륜 결함 데이터만 다루므로, 이후의 실습에서는 `DE_BPFI` 와 그 고조파(2배, 3배, …)가 포락선 스펙트럼의 **진단 지표**가 된다.
- 다만 **슬립(slip)** 때문에 실제 결함 주파수는 이론값에서 보통 1 ~ 2 % 벗어나며, 운전 조건에 따라서는 더 크게 벗어날 수 있다.

---


## 실습 2. 신호 불러오기와 파형 비교

CSV 파일 두 개를 읽어 1차원 배열로 만든다. 열 0은 샘플 인덱스이고 열 1은 가속도(g) 값이다.

- `data/data_normal.csv`: 정상 베어링(`v_n`)
- `data/data_fault_DE_IR.csv`: 내륜 결함(`v_f`)

샘플링 주파수는 CWRU 데이터의 규격인 **12 kHz** 이다. 두 신호는 길이가 10초로 같으므로 같은 시간 축을 사용할 수 있다.

In [ ]:
fs = 12000

data = np.array(pd.read_csv('./data/data_normal.csv'))
v_n = data[:, 1]

data = np.array(pd.read_csv('./data/data_fault_DE_IR.csv'))
v_f = data[:, 1]

# 시간 벡터: arange(N)/fs 로 정확히 N 개 샘플에 매핑 (오프셋 없이)
t_n = np.arange(len(v_n)) / fs
t_f = np.arange(len(v_f)) / fs

# 파형 비교 (Fault peak 기준 자동 스케일)
y_lim = np.max(np.abs(v_f)) * 1.1

plt.figure(figsize=(10, 4))
plt.plot(t_n, v_n, color='C0', label='Normal', alpha=0.9)
plt.plot(t_f, v_f, color='C1', label='Fault (DE IR)', alpha=0.7)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (g)')
plt.ylim([-y_lim, y_lim])
plt.legend()
plt.tight_layout()
plt.show()

print(f'RMS  Normal = {np.sqrt(np.mean(v_n**2)):.4f} g')
print(f'RMS  Fault  = {np.sqrt(np.mean(v_f**2)):.4f} g')
print(f'Peak Normal = {np.max(np.abs(v_n)):.4f} g')
print(f'Peak Fault  = {np.max(np.abs(v_f)):.4f} g')


**관찰**
- 결함 신호는 정상 신호에 비해 **RMS 가 약 1.2배**, **최댓값(Peak)이 약 2배**로 커진다.
- RMS 보다 최댓값의 차이가 더 크다는 점이 중요하며, 이는 결함 신호가 **충격성**을 띤다는 뜻이다.
- 그러나 시간 파형만으로는 "진폭이 조금 커졌다"는 것 외에 얻을 수 있는 정보가 부족하므로, 이제 주파수 영역에서 살펴보자.

---


## 실습 3. 스펙트럼 비교

두 신호의 단측 진폭 스펙트럼을 구하여 한 그래프에 겹쳐 비교한다. 베어링 결함에 의한 충격은 **구조 공진 대역**(수 kHz)을 주기적으로 가진하여 그 대역의 에너지를 증가시킨다. 따라서 저주파가 아니라 **고주파 공진 대역**을 먼저 확인해야 한다.

In [ ]:
f_n, A_n = utils.fft(v_n - np.mean(v_n), fs)
f_f, A_f = utils.fft(v_f - np.mean(v_f), fs)

plt.figure(figsize=(10, 4))
plt.plot(f_n, A_n, color='C0', label='Normal', alpha=0.9)
plt.plot(f_f, A_f, color='C1', label='Fault (DE IR)', alpha=0.7)
plt.xlabel('Frequency (Hz)')
plt.ylabel('|Y|')
plt.xlim([0, fs / 2])
plt.legend()
plt.tight_layout()
plt.show()


**관찰**
- 결함 신호의 스펙트럼은 **약 3 ~ 5.5 kHz 사이**의 넓은 대역에서 정상 신호보다 **열 배 이상** 크며(3.5 ~ 5.5 kHz 대역의 스펙트럼 RMS 로 약 17배), 이 대역이 구조 공진 대역이다.
- 0 ~ 500 Hz 는 그래프 폭의 1/12에 지나지 않으므로, 저주파(<500 Hz)에 있는 BPFI(≈ 160 Hz) 성분은 이 그래프에서 다른 성분과 구분되지 않는다. `plt.xlim([0, 500])` 으로 확대하면 BPFI 의 위치에 Fault 의 작은 성분이 나타나지만, 같은 구간에 이보다 큰 Fault 성분이 여섯 개 있다. 또한 Normal 에도 바로 옆의 약 161 Hz 에 이보다 조금 큰 성분이 있으므로, 원 스펙트럼에서는 BPFI 성분을 구별할 수 없다. 원 신호에 FFT 를 바로 적용하면 **결함 진단 정보가 고주파 반송파에 실려** 있으므로 저주파에서는 이 정보가 다른 성분에 묻힌다.
- 해결책은 **공진 대역만 대역통과 필터로 추출**하고 **힐베르트 포락선을 구한 뒤** 포락선을 **다시 주파수 분석**하는 것이다. 이렇게 하면 저주파에 BPFI 성분이 나타나며, 다음 실습에서 이 과정을 차례로 확인한다.

---


## 실습 4. 대역통과 필터링으로 공진 대역 분리하기

실습 3에서 확인한 공진 대역 **[3500, 5500] Hz** 만 통과시킨다. Butterworth 4차 영위상(zero-phase) 필터(`sosfiltfilt`)를 사용하므로 위상 왜곡이 없다.

필터링한 신호는 **공진 주파수로 진동하는 반송파**(carrier)가 **결함 충격의 반복 주파수**(BPFI, 약 160 Hz)로 진폭 변조된 형태가 된다. 이 AM 구조에서 변조 정보를 추출하는 다음 단계가 포락선 복조이다.

In [ ]:
f_low, f_high = 3500, 5500

v_filter_n = utils.filtering_zerophase(v_n, fs, 'band', f_low=f_low, f_high=f_high)
v_filter_f = utils.filtering_zerophase(v_f, fs, 'band', f_low=f_low, f_high=f_high)

# 자동 스케일 (fault 기준)
y_lim_filter = np.max(np.abs(v_filter_f)) * 1.1

plt.figure(figsize=(10, 4))
plt.plot(t_n, v_filter_n, color='C0', label='Normal', alpha=0.9)
plt.plot(t_f, v_filter_f, color='C1', label='Fault (DE IR)', alpha=0.7)
plt.xlabel('Time (s)')
plt.ylabel('Filtered amplitude (g)')
plt.ylim([-y_lim_filter, y_lim_filter])
plt.title(f'Bandpass [{f_low}, {f_high}] Hz')
plt.legend()
plt.tight_layout()
plt.show()


**관찰**
- 필터링한 뒤에는 **Fault 파형에서 충격성 신호가 뚜렷해지며**, 각 충격은 뾰족한 피크로 나타난다. 결함 주기는 BPFI 의 주기(약 6.25 ms)이다. 그러나 충격의 세기가 축의 회전 각도에 따라 크게 달라지므로 10초를 한 화면에 나타낸 이 그래프에서는 큰 충격만 두드러진다. 6 ms 간격은 이 그래프의 폭에서 1 픽셀보다 좁으므로 각각의 충격은 구분되지 않는다.
- Normal 은 같은 대역에서 진폭이 거의 일정한 잡음 수준이다.
- 이제 이 Fault 신호에서 **포락선**(envelope)만 추출하면 고주파 반송파는 제거되고 **결함 충격의 느린 진폭 변화**(BPFI 와 축 회전에 의한 변조)가 남는다.

---


## 실습 5. 포락선 스펙트럼으로 BPFI 확인

필터링한 신호의 순시 진폭을 **힐베르트 포락선**(`utils.hilbert_envelope`)으로 구한 뒤, DC 성분을 빼고 FFT 를 적용한다.

포락선 스펙트럼의 저주파 구간(0 ~ 500 Hz)에서는 다음을 기대한다.
- **BPFI ≈ 160 Hz** 근처의 뚜렷한 성분
- 그 **2배(≈320 Hz), 3배(≈480 Hz)의 고조파**(harmonic)
- 축 회전 주파수의 정수배 성분과, BPFI 양옆에 축 회전 주파수(≈ 29.5 Hz) 간격으로 나타나는 내륜 결함 특유의 사이드밴드 성분

In [ ]:
# Hilbert envelope (utils 래퍼 사용)
v_filter_env_n = utils.hilbert_envelope(v_filter_n)
v_filter_env_f = utils.hilbert_envelope(v_filter_f)

# DC 성분 제거 후 FFT
f_env_n, A_env_n = utils.fft(v_filter_env_n - np.mean(v_filter_env_n), fs)
f_env_f, A_env_f = utils.fft(v_filter_env_f - np.mean(v_filter_env_f), fs)

plt.figure(figsize=(10, 4))
plt.plot(f_env_n, A_env_n, color='C0', label='Normal', alpha=0.9, zorder=3)
plt.plot(f_env_f, A_env_f, color='C1', label='Fault (DE IR)', alpha=0.7, zorder=2)

# shaft harmonic (검은색), BPFI harmonic (빨강): 첫 번째에만 label 달아 legend 중복 방지
# 가는 점선을 데이터 뒤(zorder=1)에 두어 피크 높이를 가리지 않게 한다
for n in range(1, 4):
    plt.axvline(n * F_shaft, color='k', linestyle=':', lw=0.8, alpha=0.5, zorder=1,
                label='Shaft harmonic' if n == 1 else None)
    plt.axvline(n * DE_BPFI, color='C3', linestyle=':', lw=0.8, alpha=0.7, zorder=1,
                label='BPFI harmonic' if n == 1 else None)

plt.xlabel('Frequency (Hz)')
plt.ylabel('|Y|')
plt.xlim([0, 600])
plt.legend()
plt.tight_layout()
plt.show()


**관찰**
- Fault 의 포락선 스펙트럼에서는 **BPFI ≈ 160 Hz** 성분이 그래프 전체에서 가장 크다(약 0.0175). 고조파인 2×BPFI ≈ 320 Hz 성분(약 0.003)은 바로 옆의 성분들보다 조금 클 뿐이다. 3×BPFI ≈ 480 Hz 성분(약 0.0015)은 잡음 바닥보다 크지만 주변의 잡음 피크와 크기가 비슷하다. 그 왼쪽의 약 453 Hz 에 있는 큰 성분은 BPFI 의 정수배가 아니다.
- 160 Hz 아래의 성분들은 축 회전 주파수(약 29.5 Hz)의 정수배(30, 59, 89, 118 Hz)와, BPFI 양옆에 축 회전 주파수 간격으로 나타나는 사이드밴드 가운데 아래쪽 성분(17, 42, 71, 101, 130 Hz)이다. 이 성분들은 내륜 결함의 충격 세기가 축 회전 주기로 변동하여 발생한 것이므로 결함을 뒷받침한다. 다만 결함의 판정은 이론이 지정한 BPFI 의 정수배 위치에서 한다.
- Normal 에는 같은 주파수 범위에 BPFI 성분이 없다.
- **대역통과 필터로 공진 대역을 분리하면 진단에 필요한 성분이 더 뚜렷하게 나타난다.** 원 신호에서 바로 포락선을 구해도 이 신호에서는 BPFI 성분이 가장 크지만, 축 회전 성분(29.6 Hz)이 BPFI 성분의 약 85 % 크기로 함께 발생하고 3×BPFI 성분은 잡음 바닥의 약 2.4배로 작아진다.
- **대역통과 필터, 힐베르트 포락선, 주파수 분석**을 차례로 적용하는 이 과정이 베어링 진단의 핵심 파이프라인이다.

---


## 실습 6. 특징 벡터 추출과 결함/정상 비율

지금까지는 그래프로 비교하였다. 실제 진단 시스템에는 **정량화된 숫자**(특징, feature)가 필요하다. `utils.feature` 는 앞의 파이프라인을 모두 수행하는 함수이다. 다만 대역통과 필터로는 인과(causal) 필터인 `utils.filtering` 을 사용하고 앞의 샘플 100개를 0으로 둔다. 특징은 크기만 사용하므로 위상 지연은 결과를 거의 바꾸지 않는다.

| 특징 그룹 | 내용 |
| --- | --- |
| `RMS / Skew / Kurt / CF` | 원 신호의 시간 영역 통계 |
| `Band1` | 원 신호에서 구한 공진 대역 `[3500, 5500] Hz` 의 RMS |
| `RMS_filter / …` | 대역통과 필터를 적용한 신호의 시간 영역 통계 |
| `Band1_env, Band2_env, Band3_env` | 포락선 스펙트럼에서 BPFI, 2×BPFI, 3×BPFI 주변의 에너지 |

**`band_from_envelope` 설계**

베어링의 결함 주파수는 슬립 때문에 이론값에서 **보통 1 ~ 2 %** 벗어나며, 운전 조건에 따라서는 더 크게 벗어날 수 있다. 회전 속도도 조금씩 변한다. 주파수 빈 하나의 값만 읽으면 성분이 인접한 빈으로 이동하였을 때 그 성분을 놓칠 수 있다. 그래서 **1배, 2배, 3배 고조파 각각에 ±10 % 주파수 대역**을 설정하고 그 안의 에너지를 계산한다.

In [ ]:
band_filter = [3500, 5500]  # 공진 대역(실습 4와 같음)

# BPFI 와 그 고조파(2배, 3배) 각각의 ±10 % 주파수 대역
band_from_envelope = np.array([
    [1 * DE_BPFI * 0.9, 1 * DE_BPFI * 1.1],
    [2 * DE_BPFI * 0.9, 2 * DE_BPFI * 1.1],
    [3 * DE_BPFI * 0.9, 3 * DE_BPFI * 1.1],
])

print('Bandpass filter range:', band_filter)
print('Envelope harmonic bands (Hz):')
print(band_from_envelope)


In [ ]:
feature_n, feature_name = utils.feature(v_n, fs, band_filter, band_from_envelope)
feature_f, feature_name = utils.feature(v_f, fs, band_filter, band_from_envelope)

# 값의 크기가 다섯 자릿수 넘게 걸치므로 세로축을 로그로 (왜도는 음수가 될 수 있어 절댓값)
x = np.arange(len(feature_name))
plt.figure(figsize=(10, 5))
plt.plot(x, np.abs(feature_n), '-o', color='C0', label='Normal')
plt.plot(x, np.abs(feature_f), '-x', color='C1', label='Fault (DE IR)')
plt.yscale('log')
plt.xticks(x, feature_name, rotation=45, ha='right')
plt.ylabel('|Feature value|')
plt.legend()
plt.tight_layout()
plt.show()

**관찰: 절댓값 비교**
- 특징의 값이 포락선 대역 특징의 약 0.00003에서 파고율의 7까지 다섯 자릿수가 넘게 분포하므로 세로축을 로그 척도로 그렸다. 로그 축에서 눈금 한 칸의 간격은 10배이다.
- 시간 통계 특징 네 개(`RMS`, `Skew`, `Kurt`, `CF`)에서는 `Skew` 를 제외하면 정상과 결함의 두 계열이 거의 같은 높이에 있다. `Skew` 는 두 값이 모두 0에 가까우므로 두 값의 간격에는 의미가 없다. 반면 공진 대역 특징과 포락선 대역 특징(`Band1`, `RMS_filter`, `Band*_env`)으로 갈수록 두 계열의 차이가 커진다.
- 특징마다 단위와 크기가 다르므로 절댓값끼리 비교하기는 어렵다. 다음 셀에서는 특징마다 "결함/정상" **비율** 하나로 나타내 본다.

In [ ]:
feature_ratio = np.abs(feature_f) / np.abs(feature_n)

plt.figure(figsize=(10, 5))
bars = plt.bar(x, feature_ratio, color='C0', width=0.6)
plt.bar_label(bars, labels=[f'{r:.1f}' if r >= 1 else f'{r:.2f}' for r in feature_ratio])
plt.axhline(y=1, color='C3', linestyle='--', label='ratio = 1')
plt.yscale('log')
plt.xticks(x, feature_name, rotation=45, ha='right')
plt.ylabel('|Fault| / |Normal|')
plt.legend()
plt.tight_layout()
plt.show()

# 숫자로도 확인
for name, r in zip(feature_name, feature_ratio):
    print(f'{name:<14s}: {r:7.2f}')

**관찰: 비율 비교(핵심)**
- 시간 통계 특징은 `RMS` 1.2, `Kurt` 1.4, `CF` 1.7로 결함 신호에서 조금 커지는 데 그친다. 충격성 지표인 `Kurt` 와 `CF` 가 `RMS` 보다 조금 더 크게 반응한다.
- 공진 대역 특징인 `Band1`(17.2)과 `RMS_filter`(17.1), 포락선 대역 특징인 `Band1_env`(35.6), `Band2_env`(21.2), `Band3_env`(11.7)는 10배 이상 반응한다. 그 가운데 BPFI 1차 대역의 `Band1_env` 가 가장 크다.
- `Skew`(0.02)와 `Skew_filter`(23.2)는 정상과 결함의 값이 모두 0에 가까우므로 비율 자체에 의미가 없다. `Skew_filter` 의 두 값은 0.0003과 −0.008이며, 0에 가까운 값끼리의 비율은 작은 차이도 크게 확대한다.
- 결론: **공진 대역 필터링과 포락선 복조가 베어링 진단의 핵심 단계**이며, 시간 통계 특징만으로는 분별력이 낮다.

---
